# Adaptive Kalman Q/R — Colab runner

This notebook:
1. Downloads MOT datasets into `.Datasets/`
2. Points at this repo (clone or upload)
3. Runs diagnose → train → eval

**Before running:** Runtime → Change runtime type → **GPU**.

Upload or clone the `motion-predictor` project so these files exist next to this notebook:
`train_adaptive_kalman.py`, `eval_adaptive_kalman.py`, `diagnose_gradients.py`, `adaptive_kalman_*.py`.

## 0. Project root (clone or upload)

Pick **one** option below.

In [ ]:
# Option A: clone from GitHub (edit URL / branch as needed)
# !git clone https://github.com/YOUR_USER/motion-predictor.git
# %cd motion-predictor

# Option B: already uploaded / Drive-mounted project
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/path/to/motion-predictor

# Option C: this notebook lives inside the project (default for local or uploaded zip)
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'train_adaptive_kalman.py').exists(), (
    f'train_adaptive_kalman.py not found in {PROJECT_ROOT}. '
    'cd into the repo or uncomment clone/Drive above.'
)
print('PROJECT_ROOT =', PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

## 1. Download & extract MOT datasets → `.Datasets/`

Downloads **MOT17** (required) and optionally **MOT20**. Builds a `val/` split from official train sequences (MOTChallenge has no labeled public val).

DanceTrack / SportsMOT are large and often gated — enable their flags and fill Drive/`gdown` IDs if you have them.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
from urllib.request import urlretrieve

# ---------------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------------
DATA_ROOT = Path('/content/.Datasets')  # Colab default; change if you prefer Drive
# DATA_ROOT = Path('/content/drive/MyDrive/.Datasets')

DOWNLOAD_MOT17 = True
DOWNLOAD_MOT20 = False          # ~ large; enable if you need it
DOWNLOAD_DANCETRACK = False     # needs your own gdown / zip URL
DOWNLOAD_SPORTSMOT = False

# Official MOTChallenge mirrors (may redirect; retry if flaky)
MOT17_URL = 'https://motchallenge.net/data/MOT17.zip'
MOT20_URL = 'https://motchallenge.net/data/MOT20.zip'

# Optional: Google Drive file IDs (gdown) if you host mirrors
DANCETRACK_GDRIVE_ID = ''  # e.g. '1abc...'
SPORTSMOT_GDRIVE_ID = ''

# Sequences held out as val from MOT17 train (standard Split-1 style)
MOT17_VAL_SEQS = {
    'MOT17-02-FRCNN', 'MOT17-02-SDP', 'MOT17-02-DPM',
    'MOT17-11-FRCNN', 'MOT17-11-SDP', 'MOT17-11-DPM',
}
MOT20_VAL_SEQS = {'MOT20-01', 'MOT20-02'}  # adjust if you prefer another split

DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT =', DATA_ROOT)


def _download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f'  already have {dest.name}')
        return dest
    print(f'  downloading {url} -> {dest} ...')
    urlretrieve(url, dest)
    print(f'  done ({dest.stat().st_size / 1e9:.2f} GB)')
    return dest


def _extract_zip(zip_path: Path, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    print(f'  extracting {zip_path.name} -> {out_dir} ...')
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(out_dir)
    print('  extract done')


def _norm_mot_layout(dataset_dir: Path):
    """Ensure dataset_dir/train exists (MOT zips unpack as MOT17/train)."""
    if (dataset_dir / 'train').is_dir():
        return dataset_dir
    # sometimes zip extracts to DATA_ROOT/MOT17/MOT17/train
    nested = dataset_dir / dataset_dir.name
    if (nested / 'train').is_dir():
        return nested
    # flat: train sits directly under DATA_ROOT after extract
    if (DATA_ROOT / 'train').is_dir() and dataset_dir.name.startswith('MOT'):
        target = DATA_ROOT / dataset_dir.name
        target.mkdir(parents=True, exist_ok=True)
        for sub in ('train', 'test'):
            src = DATA_ROOT / sub
            if src.is_dir() and not (target / sub).exists():
                shutil.move(str(src), str(target / sub))
        return target
    raise FileNotFoundError(f'Could not find train/ under {dataset_dir}')


def _make_val_split(dataset_dir: Path, val_seqs: set):
    """Move selected train sequences into dataset_dir/val."""
    train = dataset_dir / 'train'
    val = dataset_dir / 'val'
    val.mkdir(parents=True, exist_ok=True)
    if any(val.iterdir()):
        print(f'  val already populated at {val}')
        return

    # Match exact names or sequence id (MOT17-02, MOT20-01, ...)
    val_ids = set()
    for s in val_seqs:
        parts = s.split('-')
        if len(parts) >= 2:
            val_ids.add('-'.join(parts[:2]))

    moved = 0
    for name in list(os.listdir(train)):
        src = train / name
        if not src.is_dir():
            continue
        seq_id = '-'.join(name.split('-')[:2])
        if name in val_seqs or seq_id in val_ids:
            dst = val / name
            if not dst.exists():
                shutil.move(str(src), str(dst))
                moved += 1

    print(f'  moved {moved} sequences -> {val}')
    print(f'  train seqs: {len(list(train.iterdir()))}, val seqs: {len(list(val.iterdir()))}')


def setup_mot(name: str, url: str, val_seqs: set):
    ds = DATA_ROOT / name
    if (ds / 'train').is_dir() and (ds / 'val').is_dir() and any((ds / 'val').iterdir()):
        print(f'{name}: already ready at {ds}')
        return ds
    zips = DATA_ROOT / 'zips'
    zips.mkdir(exist_ok=True)
    zip_path = _download(url, zips / f'{name}.zip')
    # extract into DATA_ROOT (zip contains MOT17/ or MOT20/)
    _extract_zip(zip_path, DATA_ROOT)
    ds = _norm_mot_layout(DATA_ROOT / name)
    _make_val_split(ds, val_seqs)
    return ds


if DOWNLOAD_MOT17:
    print('\n=== MOT17 ===')
    setup_mot('MOT17', MOT17_URL, MOT17_VAL_SEQS)

if DOWNLOAD_MOT20:
    print('\n=== MOT20 ===')
    setup_mot('MOT20', MOT20_URL, MOT20_VAL_SEQS)

if DOWNLOAD_DANCETRACK:
    print('\n=== DanceTrack ===')
    !pip -q install gdown
    assert DANCETRACK_GDRIVE_ID, 'Set DANCETRACK_GDRIVE_ID'
    out = DATA_ROOT / 'DanceTrack.zip'
    !gdown --id {DANCETRACK_GDRIVE_ID} -O {out}
    _extract_zip(out, DATA_ROOT)

if DOWNLOAD_SPORTSMOT:
    print('\n=== SportsMOT ===')
    !pip -q install gdown
    assert SPORTSMOT_GDRIVE_ID, 'Set SPORTSMOT_GDRIVE_ID'
    out = DATA_ROOT / 'SportsMOT.zip'
    !gdown --id {SPORTSMOT_GDRIVE_ID} -O {out}
    _extract_zip(out, DATA_ROOT)

print('\nDatasets present:')
for p in sorted(DATA_ROOT.iterdir()):
    if p.is_dir() and p.name != 'zips':
        train_n = len(list((p / 'train').glob('*'))) if (p / 'train').exists() else 0
        val_n = len(list((p / 'val').glob('*'))) if (p / 'val').exists() else 0
        print(f'  {p.name}: train={train_n}, val={val_n}')

## 2. Install deps & path helpers

In [ ]:
# Colab already ships PyTorch; only install extras if missing
import importlib.util
import subprocess
import sys

def _ensure(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg in ('numpy', 'pandas', 'matplotlib'):
    _ensure(pkg)

import torch
from pathlib import Path

print('torch', torch.__version__)
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

DATA_ROOT = Path('/content/.Datasets')
MOT17_TRAIN = DATA_ROOT / 'MOT17' / 'train'
MOT17_VAL = DATA_ROOT / 'MOT17' / 'val'
MOT20_TRAIN = DATA_ROOT / 'MOT20' / 'train'
MOT20_VAL = DATA_ROOT / 'MOT20' / 'val'
DT_TRAIN = DATA_ROOT / 'DanceTrack' / 'train'
DT_VAL = DATA_ROOT / 'DanceTrack' / 'val'
SM_TRAIN = DATA_ROOT / 'SportsMOT' / 'train'
SM_VAL = DATA_ROOT / 'SportsMOT' / 'val'

CKPT_DIR = Path('./checkpoints/adaptive_kalman_nn_v2')
EVAL_DIR = Path('./eval/adaptive_kalman_nn_v2')
CKPT_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

assert MOT17_TRAIN.is_dir() and MOT17_VAL.is_dir(), 'Run dataset cell first (MOT17 train/val)'
print('MOT17 train/val OK')

## 3. Gradient health check (synthetic — no data needed)

In [ ]:
!python diagnose_gradients.py --synthetic --batch_size 4 --model_type transformer

## 4. Train

Uses MOT17 by default. Uncomment extra dataset flags if downloaded.

In [ ]:
cmd = f"""python train_adaptive_kalman.py \
  --mot17_train_path {MOT17_TRAIN} \
  --mot17_val_path {MOT17_VAL} \
  --model_type transformer \
  --batch_size 64 \
  --epochs 50 \
  --save_dir {CKPT_DIR} \
  --num_workers 2 \
  --r_supervise_coeff 1.0 \
  --q_gap_coeff 2.0 \
  --q_gap_trend_coeff 0.5
"""

# Optional datasets (uncomment if present):
# if MOT20_TRAIN.is_dir():
#     cmd = cmd.replace('train_adaptive_kalman.py',\
#         f'train_adaptive_kalman.py --mot20_train_path {MOT20_TRAIN} --mot20_val_path {MOT20_VAL}')
# if DT_TRAIN.is_dir():
#     cmd += f' --dancetrack_train_path {DT_TRAIN} --dancetrack_val_path {DT_VAL} --dancetrack_weight 3'
# if SM_TRAIN.is_dir():
#     cmd += f' --sportsmot_train_path {SM_TRAIN} --sportsmot_val_path {SM_VAL}'

print(cmd)
!{cmd}

## 5. Diagnose gradients on a real sample + checkpoint

In [ ]:
ckpt = CKPT_DIR / 'best_model.pth'
assert ckpt.exists(), f'Missing {ckpt} — train first'

!python diagnose_gradients.py \
  --checkpoint {ckpt} \
  --mot17_train_path {MOT17_TRAIN} \
  --sample_idx 0 \
  --batch_size 4 \
  --model_type transformer

## 6. Evaluate

In [ ]:
ckpt = CKPT_DIR / 'best_model.pth'
assert ckpt.exists(), f'Missing {ckpt}'

eval_cmd = f"""python eval_adaptive_kalman.py \
  --checkpoint {ckpt} \
  --output_dir {EVAL_DIR} \
  --mot17_val_path {MOT17_VAL} \
  --fixed_var -1 \
  --batch_size 128 \
  --plot
"""

# if MOT20_VAL.is_dir():
#     eval_cmd += f' --mot20_val_path {MOT20_VAL}'
# if DT_VAL.is_dir():
#     eval_cmd += f' --dancetrack_val_path {DT_VAL}'
# if SM_VAL.is_dir():
#     eval_cmd += f' --sportsmot_val_path {SM_VAL}'

print(eval_cmd)
!{eval_cmd}

## 7. (Optional) Peek at eval report

In [ ]:
import json
from pathlib import Path

report_path = EVAL_DIR / 'eval_report.json'
report = json.loads(report_path.read_text())

for name, block in report.get('datasets', {}).items():
    s = block['summary']
    c = block['checks']
    print(f"\n=== {name}  checks={c.get('score')} ===")
    print(f"  nll_model={s.get('nll_model'):.4f}  calib_q_gap={s.get('calib_q_gap'):.3f}")
    print(f"  corr(R,score|obs)={s.get('corr_r_score'):.3f}  corr(Q,gap)={s.get('corr_q_gap'):.3f}")
    print(f"  mean_var_q={s.get('mean_var_q'):.2e}  mean_var_r={s.get('mean_var_r'):.2e}")

if 'overall' in report:
    print('\nOVERALL', report['overall']['checks'])

## Notes

- **MOTChallenge zips** sometimes require a free account / session cookie. If `urlretrieve` fails, download MOT17.zip manually from [motchallenge.net](https://motchallenge.net/data/MOT17/), upload to Colab / Drive, set `zip_path` and call `_extract_zip` + `_make_val_split`.
- Colab disk is limited (~70GB). Prefer MOT17 only unless you mount Drive.
- Checkpoints land in `./checkpoints/adaptive_kalman_nn_v2/` — copy them to Drive if the runtime may disconnect.
- Old softplus-R checkpoints are incompatible with the fixed R head; always train `v2` from scratch.